# Vaani Track 1 — TRACE, end to end (download → train → checkpoints)

Everything is built inside this notebook, from nothing:

1. downloads the training corpus from Hugging Face (`ARTPARK-IISc/Vaani-Noise-Event-Dataset`);
2. gets the Codabench validation set (attached dataset, or downloaded with your session cookie —
   see below; optional but strongly recommended);
3. downloads the public pretrained encoders (ATST-Frame, BEATs);
4. trains the **span model** (proposals for natural clips);
5. trains two **TraceModels** (fine-tuned ATST-Frame + BEATs with boundary-aware heads;
   AdamW and Muon members of an ensemble);
6. writes every checkpoint to `/kaggle/working/checkpoints/` — download that folder and run
   the final inference locally (commands in the last cell).

No test data is used and none is needed here.

### Measured on held-out validation (trained on 4/5 of validation, scored on the 5th)

| | natural | synthetic | all |
|---|---|---|---|
| old pipeline (span model + count head) | 1.133 | 1.634 | 1.330 |
| span model, K from transcript | 1.177 | 1.841 | 1.428 |
| **TRACE, span model trained from scratch for 12 epochs (1 h)** | **1.18–1.19** | **1.98\*** | **~1.49** |
| TRACE with the 50-epoch span model | 1.19–1.20 | 1.98\* | ~1.50 |

\* synthetic clips reuse ~117 noise recordings; the held-out fold shares them with training,
so this is optimistic (leakage-free floor 1.84). The test's synthetic third very likely reuses
the same library, which is why the validation set matters so much for training.

### Setup (once)
* **Settings → Accelerator:** GPU T4 x2. **Internet:** on. Runtime ≈ 5 h (download ~1 h, span model ~1.3 h, two TraceModels ~2 h).
* **Add-ons → Secrets:**
  * `HF_TOKEN` — a Hugging Face read token from an account that has accepted the dataset
    terms. **Required.** Tick it for this notebook.
  * `CODABENCH_SESSIONID` — *optional*: the value of your logged-in Codabench `sessionid`
    cookie. Used only to download the **validation** set. Skip it if you attach the
    validation set as a Kaggle dataset instead (Add Data → your private dataset containing
    `validationMetadata.json`).

In [ ]:
# ============================== CONFIG ==============================
BRANCH        = "trace"
SPAN_EPOCHS   = 20          # span model: still improving at epoch 12 (X1); best.pt keeps the peak
TRACE_STEPS   = 3000        # per TraceModel: held-out natural peaked at step 3000
TRACE_MODELS  = [("adamw", 0), ("muon", 1)]    # (optimiser, seed) ensemble members
QUOTAS        = '{"gold":0.45,"silver":0.10,"valnat":0.10,"remix":0.25,"valsyn":0.10}'
MAX_SHARDS    = 0           # 0 = the whole corpus; a small number for a dry run
VAL_URL       = "https://www.codabench.org/datasets/download/e62b02b0-46d1-441e-aa64-e41c7d35ca67/"
WORK          = "/tmp/work"            # corpus + validation (not saved, not in the 20 GB quota)
OUT           = "/kaggle/working/checkpoints"
# =====================================================================

In [ ]:
import os, sys, glob, json, shutil, subprocess, time, zipfile
T_START = time.time()
def sh(cmd, check=True, env=None):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, env={**os.environ, **(env or {})})
    if check and r.returncode:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r.returncode

for p in ("/tmp", "/kaggle/working"):
    t, u, f = shutil.disk_usage(p)
    print(f"{p:16s} {f/2**30:7.1f} GB free")
assert shutil.disk_usage("/tmp").free > 40 * 2**30, "need >40 GB free under /tmp"

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name, "")

HF_TOKEN = secret("HF_TOKEN")
assert HF_TOKEN, "Add-ons -> Secrets: add HF_TOKEN and tick it for this notebook"
os.environ["HF_TOKEN"] = HF_TOKEN

SRC = "/tmp/v2"
shutil.rmtree(SRC, ignore_errors=True)
sh(f"git clone -q --depth 1 -b {BRANCH} https://github.com/raut7218/vaani-sed-v2.git {SRC}")
os.chdir(SRC); sys.path.insert(0, SRC)
sh("pip -q install -r requirements.txt 2>&1 | tail -1")
import torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Settings -> Accelerator -> GPU T4 x2"
# fail fast on a token without dataset access, before an hour of anything else
sh("python scripts/download_data.py --out /tmp/_probe --list-only")

## 1. Training corpus from Hugging Face
Resumable, one parquet shard at a time; clips without timestamps are skipped (neither model
trains on them). ~120 h of audio, written as FLAC under `/tmp/work/data`.

In [ ]:
t0 = time.time()
DATA = f"{WORK}/data"
sh(f"python scripts/download_data.py --out {DATA} --skip-bronze"
   + (f" --max-shards {MAX_SHARDS}" if MAX_SHARDS else ""))
recs = [json.loads(l) for l in open(f"{DATA}/manifest.jsonl") if l.strip()]
from collections import Counter
print(len(recs), "clips", Counter(r["tier"] for r in recs), f"| {(time.time()-t0)/60:.0f} min")
assert sum(r["tier"] == "gold" for r in recs) > 0, "no verified clips downloaded"

## 2. Validation set
Used for training only (natural clips, and the clean/noisy synthetic pairs that give the
re-mixer its noise library). Order of preference: an attached Kaggle dataset, then a download
with `CODABENCH_SESSIONID`, else training continues without it (expect a lower score,
mostly on synthetic clips).

In [ ]:
VAL_META = next((p for p in glob.glob("/kaggle/input/**/validationMetadata.json", recursive=True)
                 if "__MACOSX" not in p), "")
if not VAL_META:
    sid = secret("CODABENCH_SESSIONID")
    if sid:
        import requests
        zp = f"{WORK}/validation.zip"
        os.makedirs(WORK, exist_ok=True)
        try:
            with requests.get(VAL_URL, cookies={"sessionid": sid}, stream=True, timeout=120, allow_redirects=True) as r:
                r.raise_for_status()
                with open(zp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
        except Exception as e:
            print("!! validation download failed:", type(e).__name__, str(e)[:200])
            open(zp, "wb").close()
        if not zipfile.is_zipfile(zp):
            print("!! Codabench did not return a zip (expired or wrong sessionid?) - continuing WITHOUT validation data")
        else:
            with zipfile.ZipFile(zp) as z:
                z.extractall(f"{WORK}/val")
            os.remove(zp)
            VAL_META = next((p for p in glob.glob(f"{WORK}/val/**/validationMetadata.json", recursive=True)
                             if "__MACOSX" not in p), "")
if VAL_META:
    vm = json.load(open(VAL_META))
    print("validation:", VAL_META, "|", len(vm), "clips,", sum(r["syntheticData"] for r in vm), "synthetic")
else:
    print("!! no validation set: training on the Hugging Face corpus only")

## 3. Pretrained encoders (public)

In [ ]:
sh("python scripts/fetch_encoders.py --all 2>&1 | tail -3")
for f in ("checkpoints/atst_frame.ckpt", "checkpoints/BEATs_iter3_plus_AS2M.pt"):
    assert os.path.getsize(f) > 50e6, f"{f} missing or truncated"
os.makedirs(OUT, exist_ok=True)

## 4. Span model
The repo's v2 detector (frozen encoders, anchor-free span head). It supplies the proposals
natural clips are decoded from. `best.pt` is its best validation epoch.

In [ ]:
t0 = time.time()
SPAN_RUN = f"{WORK}/span"
sh(f"torchrun --standalone --nproc_per_node=2 -m src.train.train --config configs/default.yaml "
   f"--data {DATA} --out {SPAN_RUN} --fold 0 --epochs {SPAN_EPOCHS} --time-limit-h 3.0 2>&1 "
   "| grep --line-buffered -v -E 'Warning|warnings.warn|WeightNorm.apply'", env={"PYTHONPATH": SRC})
assert os.path.exists(f"{SPAN_RUN}/best.pt"), "span model wrote no best.pt - see the first traceback above"
from tracesed.predict import export_span_checkpoint
export_span_checkpoint(f"{SPAN_RUN}/best.pt", f"{OUT}/span_best.pt")   # frozen public encoders stripped (~700 MB)
print(f"span model done in {(time.time()-t0)/60:.0f} min")

## 5. TraceModels

In [ ]:
if not VAL_META:
    QUOTAS = '{"gold":0.85,"silver":0.15}'
    vm_arg = "--val-meta ''"
else:
    vm_arg = f"--val-meta '{VAL_META}'"
TRACE_CKPTS = []
for opt, seed in TRACE_MODELS:
    t0 = time.time()
    run = f"{WORK}/trace_{opt}_s{seed}"
    sh(f"torchrun --standalone --nproc_per_node 2 -m tracesed.train {vm_arg} --corpus '{DATA}' "
       f"--steps {TRACE_STEPS} --max-steps-time 3.0 --opt {opt} --seed {seed} --hold-fold -1 "
       f"--eval-every 1000 --quotas '{QUOTAS}' --bs 12 --workers 2 --out {run}",
       env={"PYTHONPATH": SRC, "OMP_NUM_THREADS": "1"})
    assert os.path.exists(f"{run}/model.pt"), f"TraceModel {opt} wrote no model.pt"
    dst = f"{OUT}/trace_{opt}_s{seed}.pt"
    shutil.copy(f"{run}/model.pt", dst); TRACE_CKPTS.append(dst)
    print(f"TraceModel {opt}/{seed} done in {(time.time()-t0)/60:.0f} min")

## 6. Export

In [ ]:
decode = {"natural": {"decoder": "thr", "kw": {"thr": 0.5, "med": 5, "min_dur": 0.1}},
          "synthetic": {"decoder": "thr", "kw": {"thr": 0.7, "med": 1, "min_dur": 0.1}}}
json.dump(decode, open(f"{OUT}/decode.json", "w"), indent=1)
info = dict(branch=BRANCH, commit=subprocess.run("git rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip(),
            span_epochs=SPAN_EPOCHS, trace_steps=TRACE_STEPS, trace_models=TRACE_MODELS, quotas=QUOTAS,
            validation_used=bool(VAL_META), hours=round((time.time() - T_START) / 3600, 2))
json.dump(info, open(f"{OUT}/RUN_INFO.json", "w"), indent=1)
# every checkpoint must load before the session ends
from src.data.labels import LabelEncoder
from tracesed.predict import load_span_model
for p in TRACE_CKPTS:
    st = torch.load(p, map_location="cpu", weights_only=False)
    assert st["step"] > 0 and len(st["model"]) > 100, p
load_span_model(f"{OUT}/span_best.pt", "checkpoints", torch.device("cpu"))
for f in sorted(os.listdir(OUT)):
    print(f"{f:28s} {os.path.getsize(f'{OUT}/{f}')/1e6:8.1f} MB")
print(json.dumps(info, indent=1))

## 7. Final inference — run locally

Download `/kaggle/working/checkpoints/` (Output tab), then on your machine:

```bash
git clone -b trace https://github.com/raut7218/vaani-sed-v2.git && cd vaani-sed-v2
pip install -r requirements.txt            # plus a torch/torchaudio build for your GPU/CPU
python scripts/fetch_encoders.py --all     # public ATST-Frame + BEATs -> ./checkpoints
cp /path/to/downloaded/checkpoints/* checkpoints/

python -m tracesed.predict \
    --ckpt checkpoints/trace_adamw_s0.pt checkpoints/trace_muon_s1.pt \
    --f0-ckpt checkpoints/span_best.pt --ckpt-dir checkpoints \
    --decode checkpoints/decode.json \
    --audio-dir /path/to/test_input_data --out submission.zip
```

`--audio-dir` is the unzipped Codabench test package: the script finds its metadata JSON
(transcripts give each clip its exact event count; `syntheticData` routes the decoder) and
every `.wav` under it. It writes `submission.zip` containing `predictions.jsonl`.